# Bài 9: Change Data Feed (CDF)

## Mục tiêu
- Bật Change Data Feed cho bảng Delta.
- Đọc **chỉ phần thay đổi** (insert/update/delete) giữa 2 version thay vì đọc lại toàn bộ bảng.
- Hiểu các cột đặc biệt: `_change_type`, `_commit_version`, `_commit_timestamp`.


## 9.1. Vấn đề CDF giải quyết

Không có CDF, muốn biết "những gì đã thay đổi kể từ lần đồng bộ trước" giữa 2 hệ thống (ví dụ đẩy dữ liệu từ bảng silver sang 1 datamart khác), ta buộc phải:
- So sánh toàn bộ snapshot cũ và mới (tốn kém), hoặc
- Tự cài đặt cơ chế theo dõi thay đổi riêng.

**Change Data Feed** để Delta Lake tự ghi lại **record-level changes** (thêm, sửa, xoá) mỗi khi có `UPDATE`/`DELETE`/`MERGE`/streaming write, giúp đọc incremental hiệu quả — nền tảng cho CDC (Change Data Capture) pipelines.

## 9.2. Bật CDF

```sql
ALTER TABLE db.tbl SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```
Hoặc khi tạo bảng:
```sql
CREATE TABLE db.tbl (...) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true);
```
> CDF chỉ ghi lại thay đổi **từ thời điểm bật trở đi** — các version trước khi bật không có change data.

## 9.3. Đọc change feed

```sql
SELECT * FROM table_changes('db.tbl', 2, 5);              -- tu version 2 den version 5
SELECT * FROM table_changes('db.tbl', '2024-01-01', '2024-01-31');  -- theo khoang thoi gian
```
Hoặc DataFrameReader:
```python
spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 2) \
    .option("endingVersion", 5) \
    .table("db.tbl")
```

Kết quả trả về **tất cả cột gốc** cộng thêm 3 cột metadata:
- `_change_type`: `insert`, `update_preimage` (giá trị *trước* update), `update_postimage` (giá trị *sau* update), `delete`.
- `_commit_version`: version tạo ra thay đổi này.
- `_commit_timestamp`: thời điểm commit.

Một `UPDATE` sinh ra **2 dòng** trong change feed cho 1 record bị sửa: 1 dòng `update_preimage` + 1 dòng `update_postimage` — cho phép biết chính xác giá trị trước/sau.


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai09"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai09-cdf")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 9.4. Ví dụ minh hoạ

In [ ]:
spark.sql("DROP TABLE IF EXISTS bai09.customers")
spark.sql("""
CREATE TABLE bai09.customers (id INT, name STRING, tier STRING)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql("INSERT INTO bai09.customers VALUES (1,'An','silver'), (2,'Binh','silver')")   # version 1 (0 la CREATE)
spark.sql("UPDATE bai09.customers SET tier = 'gold' WHERE id = 1")                        # version 2
spark.sql("DELETE FROM bai09.customers WHERE id = 2")                                     # version 3
spark.sql("INSERT INTO bai09.customers VALUES (3, 'Chi', 'silver')")                      # version 4

spark.sql("DESCRIBE HISTORY bai09.customers").select("version","operation").show()


In [ ]:
# Doc toan bo thay doi tu version 1 den version 4
changes = spark.sql("SELECT * FROM table_changes('bai09.customers', 1, 4)")
changes.orderBy("_commit_version").show(truncate=False)


In [ ]:
# Tuong duong bang DataFrameReader
changes2 = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("bai09.customers")
)
changes2.orderBy("_commit_version").show(truncate=False)


## 9.5. Thực hành

**Bài 1** — Tạo bảng `bai09.products` với `delta.enableChangeDataFeed = true` ngay khi tạo. Thực hiện 1 `INSERT`, 1 `UPDATE`, 1 `DELETE`.

**Bài 2** — Đọc toàn bộ change feed bằng `table_changes(...)`. Xác định dòng nào là `insert`, `update_preimage`, `update_postimage`, `delete`.

**Bài 3** — Chỉ lấy các thay đổi loại `update_postimage` (giá trị sau khi update) — dùng để biết "trạng thái mới nhất của các record vừa bị sửa" mà không cần đọc lại toàn bộ bảng.

**Bài 4** — Tạo 1 bảng khác `bai09.products_legacy` **không** bật CDF, thử đọc change feed của nó — quan sát lỗi xảy ra và giải thích.

**Bài 5 (tư duy)** — So sánh CDF với việc tự thêm cột `updated_at`/`is_deleted` (soft delete) vào bảng rồi tự query `WHERE updated_at > last_sync_time`. CDF có ưu điểm gì mà cách tự làm khó đạt được (gợi ý: nghĩ về trường hợp 1 record bị update nhiều lần trong cùng ngày, hoặc bị xoá thật — không phải soft delete)?


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

_Viết câu trả lời của bạn ở đây._

---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
spark.sql("DROP TABLE IF EXISTS bai09.products")
spark.sql("""
CREATE TABLE bai09.products (id INT, name STRING, price DOUBLE)
USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql("INSERT INTO bai09.products VALUES (1,'Keyboard',25.0),(2,'Mouse',10.0)")
spark.sql("UPDATE bai09.products SET price = 30.0 WHERE id = 1")
spark.sql("DELETE FROM bai09.products WHERE id = 2")


In [ ]:
# Dap an Bai 2
cdf = spark.sql("SELECT * FROM table_changes('bai09.products', 0, 10)")
cdf.orderBy("_commit_version").show(truncate=False)


In [ ]:
# Dap an Bai 3
spark.sql("SELECT * FROM table_changes('bai09.products', 0, 10) WHERE _change_type = 'update_postimage'").show(truncate=False)


In [ ]:
# Dap an Bai 4
spark.sql("DROP TABLE IF EXISTS bai09.products_legacy")
spark.sql("CREATE TABLE bai09.products_legacy (id INT, name STRING) USING DELTA")
spark.sql("INSERT INTO bai09.products_legacy VALUES (1,'X')")
try:
    spark.sql("SELECT * FROM table_changes('bai09.products_legacy', 0, 1)").show()
except Exception as e:
    print("Loi nhu du kien - CDF chua duoc bat cho bang nay:", type(e).__name__)


**Đáp án Bài 5**: Với cách tự thêm `updated_at`, nếu 1 record bị update 3 lần trong ngày, downstream chỉ query `WHERE updated_at > last_sync` sẽ **chỉ thấy giá trị cuối cùng**, mất hoàn toàn thông tin về 2 lần update trung gian (có thể quan trọng cho audit/CDC đúng nghĩa). Với record bị **xoá thật** (`DELETE`, không phải soft-delete), cách tự làm **không có cách nào biết được** record đó từng tồn tại và vừa bị xoá, trừ khi kỷ luật dùng soft-delete cho mọi bảng (tốn thêm logic ở mọi nơi ghi dữ liệu, và bảng phình to vì không bao giờ xoá thật). CDF ghi lại **từng sự kiện thay đổi** (kể cả các postimage/preimage trung gian và cả DELETE thật) một cách nhất quán, tự động, không cần sửa logic ghi dữ liệu ở tầng ứng dụng.
